# WorldKnower

Predict the country of a GeoGuessr screenshot with a fine-tuned ResNet18.

**How to use:** set the runtime to GPU (Runtime → Change runtime type → T4 GPU), then run the cells from the top.
Stop before the **Training** section unless you want to train a new model.

Data, splits, checkpoints, and figures live on the shared Drive (`WorldKnowerDrive`). Code comes from GitHub, so
merged changes show up every time you run the clone cell. Anything saved only under `/content/` is wiped when the runtime resets.

## Setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO = "/content/WorldKnower"
BRANCH = "main"
if not os.path.exists(REPO):
    !git clone https://github.com/TechBroMoho/WorldKnower.git {REPO}
!git -C {REPO} fetch -q origin
!git -C {REPO} checkout -q {BRANCH}
!git -C {REPO} pull -q origin {BRANCH}
!git -C {REPO} log --oneline -1
%cd {REPO}

## Data

In [ ]:
import os, glob, pandas as pd
DRIVE = "/content/drive/MyDrive/WorldKnowerDrive/Data"
LOCAL = "/content/data"
os.makedirs(LOCAL, exist_ok=True)

# Copy the zip to local disk first (much faster to read than Drive), then unzip
if not glob.glob(f"{LOCAL}/**/compressed_dataset", recursive=True):
    !cp "{DRIVE}/raw/worldknower_dataset.zip" /content/
    !unzip -q -n /content/worldknower_dataset.zip -d {LOCAL}

IMAGE_ROOT = glob.glob(f"{LOCAL}/**/compressed_dataset", recursive=True)[0]
print("Image root:", IMAGE_ROOT, "| countries:", len(os.listdir(IMAGE_ROOT)))

# Rewrite split CSVs as path,country with paths relative to IMAGE_ROOT
os.makedirs("/content/splits", exist_ok=True)
for name in ["train", "val", "test"]:
    df = pd.read_csv(f"{DRIVE}/splits/world_knower_{name}.csv")
    df["path"] = df["image_path"].str.replace("\\", "/", regex=False).str.split("compressed_dataset/").str[-1]
    df[["path", "country"]].to_csv(f"/content/splits/{name}.csv", index=False)
    missing = sum(not os.path.exists(os.path.join(IMAGE_ROOT, p)) for p in df["path"])
    print(f"{name}: {len(df)} rows, {df.country.nunique()} countries, {missing} missing images")

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/WorldKnowerDrive/checkpoints/baseline_r18"
RESULTS_DIR = "/content/drive/MyDrive/WorldKnowerDrive/results/figures"

## Evaluate

Scores a saved checkpoint on the validation split. Keep the test split for the final model.

In [ ]:
%cd {REPO}
!python src/evaluate.py \
    --checkpoint "{CKPT_DIR}/best_resnet18.pth" \
    --split-csv /content/splits/val.csv \
    --dataset-root "{IMAGE_ROOT}"

# evaluate.py saves the figure inside the repo, which is wiped on reset, so copy it to Drive
!mkdir -p "{RESULTS_DIR}"
!cp results/figures/confusion_matrix.png "{RESULTS_DIR}/"
!ls -lh "{RESULTS_DIR}"

## ⚠️ Training

Only run this to train a **new** model. It writes to `CKPT_DIR`, so change that folder name first
(e.g. `baseline_r18` → `frozen_r18`) or it will overwrite the existing checkpoint.

In [ ]:
%cd {REPO}
!python -u -m src.train --device cuda \
    --train-csv /content/splits/train.csv --val-csv /content/splits/val.csv \
    --dataset-root "{IMAGE_ROOT}" \
    --checkpoint-dir "{CKPT_DIR}" --history-path "{CKPT_DIR}/history.csv" \
    --epochs 10 --lr 1e-4 --batch-size 32 --num-workers 2